<a href="https://colab.research.google.com/github/blessingchidozie/INTRUDER-PROJECT/blob/main/INTRUDER_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================
# Intruder / Visitor Recognition System
# Google Colab - Ready to Run
# ======================================

# Step 1: Install dependencies
!pip install face_recognition cmake dlib --quiet
!pip install opencv-python --quiet

import face_recognition
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
import os

# Step 2: Create folders for reference images
!mkdir -p known_faces
!mkdir -p test_images

# Upload your reference images for Mum, Me, Sis Pes, Sis Mess, Bro Wis
# Use one clear image per person (frontal face, good lighting)
print("⬆️ Upload your reference images for known people into 'known_faces' folder")
print("📌 Example filenames: mum.jpg, me.jpg, sis_pes.jpg, sis_mess.jpg, bro_wis.jpg")

from google.colab import files
uploaded = files.upload()

for fn in uploaded.keys():
    with open(os.path.join("known_faces", fn), 'wb') as f:
        f.write(uploaded[fn])

# Step 3: Load known faces
known_face_encodings = []
known_face_names = []

for filename in os.listdir("known_faces"):
    image_path = os.path.join("known_faces", filename)
    image = face_recognition.load_image_file(image_path)
    encoding = face_recognition.face_encodings(image)
    if encoding:
        known_face_encodings.append(encoding[0])
        # Name from file without extension
        known_face_names.append(os.path.splitext(filename)[0])
    else:
        print(f"⚠️ No face found in {filename}")

print("✅ Loaded known faces:", known_face_names)

# Step 4: Upload a visitor/test image
print("\n⬆️ Upload an image of the visitor to 'test_images' folder")
visitor_upload = files.upload()
for fn in visitor_upload.keys():
    with open(os.path.join("test_images", fn), 'wb') as f:
        f.write(visitor_upload[fn])

# Step 5: Recognition
for filename in os.listdir("test_images"):
    img_path = os.path.join("test_images", filename)
    unknown_image = face_recognition.load_image_file(img_path)

    face_locations = face_recognition.face_locations(unknown_image)
    face_encodings = face_recognition.face_encodings(unknown_image, face_locations)

    image_bgr = cv2.cvtColor(unknown_image, cv2.COLOR_RGB2BGR)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)
        name = "Unknown"

        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        best_match_index = np.argmin(face_distances)
        if matches[best_match_index]:
            name = known_face_names[best_match_index]

        # Draw box
        cv2.rectangle(image_bgr, (left, top), (right, bottom), (0, 255, 0), 2)
        # Draw label
        cv2.rectangle(image_bgr, (left, bottom - 25), (right, bottom), (0, 255, 0), cv2.FILLED)
        cv2.putText(image_bgr, name, (left + 6, bottom - 6),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

    print(f"Result for {filename}:")
    cv2_imshow(image_bgr)

print("\n🎯 Recognition Complete!")


RuntimeError: Error while calling cudaGetDevice(&the_device_id) in file /tmp/.tmpNZoFGg/sdists-v9/pypi/dlib/19.24.6/mZZaFg42M-YV0ZsyCth_2/src/dlib/cuda/gpu_data.cpp:204. code: 35, reason: CUDA driver version is insufficient for CUDA runtime version

In [ ]:
# ======================================
# Intruder / Visitor Recognition System
# Google Colab - Live Webcam Version
# ======================================

# Step 1: Install dependencies
!pip install face_recognition cmake dlib --quiet
!pip install opencv-python matplotlib --quiet

import face_recognition
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import files, output
import io
import PIL.Image

# Step 2: Create folders for known faces
!mkdir -p known_faces

# Upload your reference images for Mum, Me, Sis Pes, Sis Mess, Bro Wis
print("⬆️ Upload your reference images for known people into 'known_faces' folder")
print("📌 Example filenames: mum.jpg, me.jpg, sis_pes.jpg, sis_mess.jpg, bro_wis.jpg")
uploaded = files.upload()
for fn in uploaded.keys():
    with open(os.path.join("known_faces", fn), 'wb') as f:
        f.write(uploaded[fn])

# Step 3: Load known faces
known_face_encodings = []
known_face_names = []

for filename in os.listdir("known_faces"):
    image_path = os.path.join("known_faces", filename)
    image = face_recognition.load_image_file(image_path)
    encoding = face_recognition.face_encodings(image)
    if encoding:
        known_face_encodings.append(encoding[0])
        known_face_names.append(os.path.splitext(filename)[0])
    else:
        print(f"⚠️ No face found in {filename}")

print("✅ Loaded known faces:", known_face_names)

# Step 4: Webcam capture helper
def take_photo(filename='visitor.jpg', quality=0.9):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize output to fit screen
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    takePhoto({quality: %f}).then(dataUrl => {
      const data = dataUrl.split(',')[1];
      google.colab.kernel.invokeFunction('notebook.capture', [data], {});
    });
    ''' % quality)
    display(js)

def capture_callback(data):
    with open(filename, "wb") as f:
        f.write(base64.b64decode(data))

from IPython.display import Javascript, display
import base64
output.register_callback('notebook.capture', capture_callback)

# Step 5: Capture image from webcam
print("📷 Starting webcam... Click 'Capture' to take a photo.")
take_photo('visitor.jpg')

# Step 6: Recognition from webcam photo
unknown_image = face_recognition.load_image_file('visitor.jpg')
face_locations = face_recognition.face_locations(unknown_image)
face_encodings = face_recognition.face_encodings(unknown_image, face_locations)

image_bgr = cv2.cvtColor(unknown_image, cv2.COLOR_RGB2BGR)

for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)
    name = "Unknown"

    face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
    best_match_index = np.argmin(face_distances)
    if matches[best_match_index]:
        name = known_face_names[best_match_index]

    # Draw box + label
    cv2.rectangle(image_bgr, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.rectangle(image_bgr, (left, bottom - 25), (right, bottom), (0, 255, 0), cv2.FILLED)
    cv2.putText(image_bgr, name, (left + 6, bottom - 6),
                cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

# Step 7: Display result
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Recognition Result")
plt.show()

print("🎯 Recognition complete!")


RuntimeError: Error while calling cudaGetDevice(&the_device_id) in file /tmp/.tmpNZoFGg/sdists-v9/pypi/dlib/19.24.6/mZZaFg42M-YV0ZsyCth_2/src/dlib/cuda/gpu_data.cpp:204. code: 35, reason: CUDA driver version is insufficient for CUDA runtime version

In [ ]:
# ======================================
# Intruder / Visitor Recognition System
# Google Colab - Live Webcam Version (CPU Mode)
# ======================================

# Step 1: Install dependencies (CPU build only)
!pip uninstall -y dlib face_recognition > /dev/null
!pip install face_recognition --no-binary :all: --no-cache-dir --quiet
!pip install opencv-python matplotlib --quiet

import face_recognition
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import files, output
from IPython.display import Javascript, display
import base64
import io
import PIL.Image

# Step 2: Create folders for known faces
!mkdir -p known_faces

# Upload your reference images for Mum, Me, Sis Pes, Sis Mess, Bro Wis
print("⬆️ Upload your reference images for known people into 'known_faces' folder")
print("📌 Example filenames: mum.jpg, me.jpg, sis_pes.jpg, sis_mess.jpg, bro_wis.jpg")
uploaded = files.upload()
for fn in uploaded.keys():
    with open(os.path.join("known_faces", fn), 'wb') as f:
        f.write(uploaded[fn])

# Step 3: Load known faces
known_face_encodings = []
known_face_names = []

for filename in os.listdir("known_faces"):
    image_path = os.path.join("known_faces", filename)
    image = face_recognition.load_image_file(image_path)
    encoding = face_recognition.face_encodings(image)
    if encoding:
        known_face_encodings.append(encoding[0])
        known_face_names.append(os.path.splitext(filename)[0])
    else:
        print(f"⚠️ No face found in {filename}")

print("✅ Loaded known faces:", known_face_names)

# Step 4: Webcam capture helper
def take_photo(filename='visitor.jpg', quality=0.9):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Adjust iframe height
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    takePhoto({quality: %f}).then(dataUrl => {
      const data = dataUrl.split(',')[1];
      google.colab.kernel.invokeFunction('notebook.capture', [data], {});
    });
    ''' % quality)
    display(js)

def capture_callback(data):
    with open('visitor.jpg', "wb") as f:
        f.write(base64.b64decode(data))

output.register_callback('notebook.capture', capture_callback)

# Step 5: Capture image from webcam
print("📷 Starting webcam... Click 'Capture' to take a photo.")
take_photo('visitor.jpg')

# Step 6: Recognition from webcam photo
unknown_image = face_recognition.load_image_file('visitor.jpg')
face_locations = face_recognition.face_locations(unknown_image)
face_encodings = face_recognition.face_encodings(unknown_image, face_locations)

image_bgr = cv2.cvtColor(unknown_image, cv2.COLOR_RGB2BGR)

for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)
    name = "Unknown"

    face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
    best_match_index = np.argmin(face_distances)
    if matches[best_match_index]:
        name = known_face_names[best_match_index]

    # Draw box + label
    cv2.rectangle(image_bgr, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.rectangle(image_bgr, (left, bottom - 25), (right, bottom), (0, 255, 0), cv2.FILLED)
    cv2.putText(image_bgr, name, (left + 6, bottom - 6),
                cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

# Step 7: Display result
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Recognition Result")
plt.show()

print("🎯 Recognition complete! (CPU Mode)")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 271.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (dlib)


ModuleNotFoundError: No module named 'face_recognition'

In [ ]:
# =========================
# STEP 1 — Install Packages
# =========================
!pip install face_recognition cmake
!pip install opencv-python

# =========================
# STEP 2 — Import Libraries
# =========================
import face_recognition
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow

# =========================
# STEP 3 — Upload Dataset
# =========================
from google.colab import files
import shutil

# Create main folder
if not os.path.exists("dataset"):
    os.makedirs("dataset")

# Categories
categories = ["mum", "me", "sis_pes", "sis_mess", "bro_wis", "others"]

for category in categories:
    path = f"dataset/{category}"
    os.makedirs(path, exist_ok=True)
    print(f"📂 Upload images for: {category}")
    uploaded = files.upload()
    for filename in uploaded.keys():
        shutil.move(filename, os.path.join(path, filename))

print("\n✅ All images uploaded!")

# =========================
# STEP 4 — Encode Faces
# =========================
known_encodings = []
known_names = []

for category in categories:
    folder = f"dataset/{category}"
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        image = face_recognition.load_image_file(img_path)
        encodings = face_recognition.face_encodings(image)
        if len(encodings) > 0:
            known_encodings.append(encodings[0])
            known_names.append(category)
        else:
            print(f"⚠ No face found in {filename} — skipped.")

print(f"\n✅ Encoded {len(known_encodings)} faces.")

# =========================
# STEP 5 — Upload Test Image
# =========================
print("\n📷 Upload test image...")
test_file = files.upload()
test_filename = list(test_file.keys())[0]
test_image = face_recognition.load_image_file(test_filename)
test_encodings = face_recognition.face_encodings(test_image)

image_bgr = cv2.imread(test_filename)

# =========================
# STEP 6 — Recognition
# =========================
for face_encoding, face_location in zip(test_encodings, face_recognition.face_locations(test_image)):
    matches = face_recognition.compare_faces(known_encodings, face_encoding, tolerance=0.5)
    name = "Unknown"

    if True in matches:
        first_match_index = matches.index(True)
        name = known_names[first_match_index]

    top, right, bottom, left = face_location
    cv2.rectangle(image_bgr, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(image_bgr, name, (left, top-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

cv2_imshow(image_bgr)
print("\n🎯 Recognition Complete!")


  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
Failed to build dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (dlib)


ModuleNotFoundError: No module named 'face_recognition'

In [ ]:
!pip install face_recognition

import face_recognition
import cv2
import os
import numpy as np

# === Step 1: Load known faces ===
known_face_encodings = []
known_face_names = []

known_faces_dir = "known_faces"  # folder with subfolders per person
for name in os.listdir(known_faces_dir):
    person_folder = os.path.join(known_faces_dir, name)
    if not os.path.isdir(person_folder):
        continue
    for filename in os.listdir(person_folder):
        img_path = os.path.join(person_folder, filename)
        img = face_recognition.load_image_file(img_path)
        encoding = face_recognition.face_encodings(img)
        if encoding:
            known_face_encodings.append(encoding[0])
            known_face_names.append(name)

print(f"[INFO] Loaded {len(known_face_encodings)} known faces.")

# === Step 2: Start video capture ===
video = cv2.VideoCapture(0)  # 0 = webcam

while True:
    ret, frame = video.read()
    if not ret:
        break

    # Reduce size for speed (1/4)
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # Detect faces (HOG = fast on CPU)
    face_locations = face_recognition.face_locations(rgb_small_frame, model="hog")
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    face_names = []
    for face_encoding in face_encodings:
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.45)
        name = "Unknown"

        # Use smallest distance
        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        best_match_index =


SyntaxError: invalid syntax (ipython-input-4153603136.py, line 50)

In [ ]:
!pip install face_recognition opencv-python matplotlib --quiet



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (dlib)


In [ ]:
import face_recognition
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

# Load image
image_path = "your_image.jpg"  # change this to your image path
image = face_recognition.load_image_file(image_path)

# Start timing
start = time.time()

# Detect faces using HOG model (fast on CPU)
face_locations = face_recognition.face_locations(image, model="hog")

# End timing
end = time.time()

print(f"Found {len(face_locations)} face(s) in {end - start:.2f} seconds")

# Draw boxes
image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
for (top, right, bottom, left) in face_locations:
    cv2.rectangle(image_bgr, (left, top), (right, bottom), (0, 255, 0), 2)

# Show image
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


ModuleNotFoundError: No module named 'face_recognition'

In [ ]:
!pip install face_recognition opencv-python matplotlib --quiet

import face_recognition
import cv2
import time

# Load image
image = face_recognition.load_image_file("your_image.jpg")

# Start timing
start_time = time.time()

# Detect faces using faster HOG model
face_locations = face_recognition.face_locations(image, model="hog")

# End timing
end_time = time.time()

print(f"Found {len(face_locations)} face(s) in {end_time - start_time:.2f} seconds.")

# Draw boxes
image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
for (top, right, bottom, left) in face_locations:
    cv2.rectangle(image_bgr, (left, top), (right, bottom), (0, 255, 0), 2)

cv2.imwrite("output.jpg", image_bgr)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (dlib)


ModuleNotFoundError: No module named 'face_recognition'

In [ ]:
# Install dlib and face_recognition
!apt-get install -y cmake
!pip install dlib==19.24.2 face_recognition


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 110.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
Failed to build dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (dlib)


In [ ]:
# Remove any broken installs
!pip uninstall -y face_recognition dlib

# Install CMake (needed for building)
!apt-get install -y cmake

# Install precompiled dlib (CPU only)
!pip install dlib-bin

# Now install face_recognition
!pip install face_recognition



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 33.4 MB/s eta 0:00:00
  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
  Using cached dlib-20.0.0.tar.gz (3.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
Failed to build dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml

In [ ]:
# Remove any broken installs
!pip uninstall -y face_recognition dlib

# Install prebuilt dlib (CPU only, no compilation)
!pip install dlib-bin==19.24.6

# Install face_recognition but tell pip NOT to upgrade dlib
!pip install face_recognition --no-deps


  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)


In [ ]:
import face_recognition
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow


In [ ]:
# ===============================================
# 1. Install dependencies
# ===============================================
!pip install face_recognition dlib-bin

# ===============================================
# 2. Imports
# ===============================================
import face_recognition
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow
from zipfile import ZipFile

# ===============================================
# 3. Upload your dataset.zip
# ===============================================
from google.colab import files
uploaded = files.upload()

# ===============================================
# 4. Unzip dataset
# ===============================================
with ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

# ===============================================
# 5. Load known faces
# ===============================================
known_face_encodings = []
known_face_names = []

dataset_path = 'dataset'

for person_name in os.listdir(dataset_path):
    person_folder = os.path.join(dataset_path, person_name)
    if os.path.isdir(person_folder):
        for image_name in os.listdir(person_folder):
            image_path = os.path.join(person_folder, image_name)
            image = face_recognition.load_image_file(image_path)
            encoding = face_recognition.face_encodings(image)
            if encoding:
                known_face_encodings.append(encoding[0])
                known_face_names.append(person_name)

print("✅ Loaded faces for:", known_face_names)

# ===============================================
# 6. Test with a new image
# ===============================================
print("📷 Upload a test image (e.g., intruder photo)")
uploaded_test = files.upload()

test_image_path = list(uploaded_test.keys())[0]
test_image = face_recognition.load_image_file(test_image_path)

face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

# Convert to BGR for OpenCV display
test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
    name = "Unknown"

    face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
    best_match_index = np.argmin(face_distances)
    if matches[best_match_index]:
        name = known_face_names[best_match_index]

    cv2.rectangle(test_image_cv, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(test_image_cv, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

cv2_imshow(test_image_cv)
cv2.waitKey(0)
cv2.destroyAllWindows()


  Using cached dlib-20.0.0.tar.gz (3.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
Failed to build dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (dlib)


In [ ]:
import os
os.listdir()


In [ ]:
import zipfile

with zipfile.ZipFile("faces_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("faces_dataset")


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
# 1. Install dependencies with prebuilt binaries
!pip install cmake
!pip install dlib-bin==19.24.6
!pip install face_recognition

# 2. Verify installation
import face_recognition
print("face_recognition is installed and working ✅")

# 3. Upload your dataset zip
from google.colab import files
uploaded = files.upload()

# 4. Unzip the dataset (replace with your actual uploaded filename)
import zipfile
zip_path = "faces_dataset.zip"  # make sure it matches your uploaded file name
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("faces_dataset")

print("Dataset extracted successfully ✅")


In [ ]:
# ============================================
# 1. INSTALL DEPENDENCIES
# ============================================
!apt-get update
!apt-get install -y cmake
!pip install face_recognition opencv-python

# ============================================
# 2. IMPORT LIBRARIES
# ============================================
import face_recognition
import cv2
import numpy as np
import os
import zipfile

# ============================================
# 3. UPLOAD & UNZIP DATASET
# ============================================
from google.colab import files
print("Upload faces_dataset.zip (contains folders with person names)...")
uploaded = files.upload()

# Extract the uploaded zip
zip_path = "faces_dataset.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("faces_dataset")

print("Dataset extracted to faces_dataset/")

# ============================================
# 4. LOAD AND ENCODE KNOWN FACES
# ============================================
known_face_encodings = []
known_face_names = []

for person_name in os.listdir("faces_dataset"):
    person_folder = os.path.join("faces_dataset", person_name)
    if not os.path.isdir(person_folder):
        continue

    for filename in os.listdir(person_folder):
        img_path = os.path.join(person_folder, filename)
        try:
            image = face_recognition.load_image_file(img_path)
            encoding = face_recognition.face_encodings(image)[0]
            known_face_encodings.append(encoding)
            known_face_names.append(person_name)
        except IndexError:
            print(f"⚠ No face found in {img_path}, skipping.")

print(f"Loaded {len(known_face_names)} known faces.")

# ============================================
# 5. TEST RECOGNITION ON AN IMAGE
# ============================================
print("Upload a test image for recognition...")
test_upload = files.upload()
test_image_path = list(test_upload.keys())[0]

# Load test image
test_image = face_recognition.load_image_file(test_image_path)
face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

# Convert to OpenCV format for display
test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
    face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
    best_match_index = np.argmin(face_distances)

    if matches[best_match_index]:
        name = known_face_names[best_match_index]
    else:
        name = "Unknown"

    # Draw box and label
    cv2.rectangle(test_image_cv, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(test_image_cv, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# ============================================
# 6. SHOW RESULT
# ============================================
from google.colab.patches import cv2_imshow
cv2_imshow(test_image_cv)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
# ============================
# STEP 1: Install dependencies
# ============================
!apt-get update
!apt-get install -y cmake
!pip install face_recognition opencv-python

# ============================
# STEP 2: Import libraries
# ============================
import os
import zipfile
import face_recognition
import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# ============================
# STEP 3: Upload & unzip dataset
# ============================
print("⬆️ Upload your faces_dataset.zip (with subfolders per person)")
uploaded = files.upload()

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("faces_dataset")

print("✅ Dataset extracted to 'faces_dataset/'")

# ============================
# STEP 4: Load and encode known faces
# ============================
known_face_encodings = []
known_face_names = []

dataset_dir = "faces_dataset"

for person_name in os.listdir(dataset_dir):
    person_folder = os.path.join(dataset_dir, person_name)
    if not os.path.isdir(person_folder):
        continue

    for filename in os.listdir(person_folder):
        file_path = os.path.join(person_folder, filename)
        if os.path.isfile(file_path) and file_path.lower().endswith(('.jpg', '.jpeg', '.png')):
            image = face_recognition.load_image_file(file_path)
            encodings = face_recognition.face_encodings(image)
            if encodings:
                known_face_encodings.append(encodings[0])
                known_face_names.append(person_name)
            else:
                print(f"⚠️ No face found in {file_path}, skipping.")

print(f"✅ Encoded {len(known_face_names)} faces from dataset.")

# ============================
# STEP 5: Upload test image
# ============================
print("⬆️ Upload a test image to recognize faces in...")
test_uploaded = files.upload()

test_image_path = list(test_uploaded.keys())[0]

test_image = face_recognition.load_image_file(test_image_path)
face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

# Convert to OpenCV BGR format for display
test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

# ============================
# STEP 6: Run recognition & display
# ============================
for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)
    face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
    best_match_index = np.argmin(face_distances)

    if matches and matches[best_match_index]:
        name = known_face_names[best_match_index]
    else:
        name = "Unknown"

    cv2.rectangle(test_image_cv, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(test_image_cv, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

cv2_imshow(test_image_cv)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("\n🎯 Recognition complete!")


In [ ]:
# ============================
# STEP 1: Install dependencies
# ============================
!apt-get update
!apt-get install -y cmake
!pip install face_recognition opencv-python

# ============================
# STEP 2: Import libraries
# ============================
import os
import zipfile
import face_recognition
import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# ============================
# STEP 3: Upload & unzip dataset
# ============================
print("⬆️ Upload your faces_dataset.zip (with subfolders per person)")
uploaded = files.upload()

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("faces_dataset")

print("✅ Dataset extracted to 'faces_dataset/'")

# ============================
# STEP 4: Load and encode known faces
# ============================
known_face_encodings = []
known_face_names = []

dataset_dir = "faces_dataset"

for person_name in os.listdir(dataset_dir):
    person_folder = os.path.join(dataset_dir, person_name)
    if not os.path.isdir(person_folder):
        continue

    for filename in os.listdir(person_folder):
        file_path = os.path.join(person_folder, filename)
        if os.path.isfile(file_path) and file_path.lower().endswith(('.jpg', '.jpeg', '.png')):
            image = face_recognition.load_image_file(file_path)
            encodings = face_recognition.face_encodings(image)
            if encodings:
                known_face_encodings.append(encodings[0])
                known_face_names.append(person_name)
            else:
                print(f"⚠️ No face found in {file_path}, skipping.")

print(f"✅ Encoded {len(known_face_names)} faces from dataset.")

# ============================
# STEP 5: Upload test image
# ============================
print("⬆️ Upload a test image to recognize faces in...")
test_uploaded = files.upload()

test_image_path = list(test_uploaded.keys())[0]

test_image = face_recognition.load_image_file(test_image_path)
face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

# Convert to OpenCV BGR format for display
test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

# ============================
# STEP 6: Run recognition & display
# ============================
for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)
    face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
    best_match_index = np.argmin(face_distances)

    if matches and matches[best_match_index]:
        name = known_face_names[best_match_index]
    else:
        name = "Unknown"

    cv2.rectangle(test_image_cv, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(test_image_cv, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

cv2_imshow(test_image_cv)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("\n🎯 Recognition complete!")


In [ ]:
# ============================
# STEP 1: Install dependencies
# ============================
!apt-get update
!apt-get install -y cmake
!pip install face_recognition opencv-python

# ============================
# STEP 2: Import libraries
# ============================
import os
import zipfile
import face_recognition
import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# ============================
# STEP 3: Upload & unzip dataset
# ============================
print("⬆️ Upload your faces_dataset.zip (with subfolders: mum, me, sis_pec, sis_mess, bro_wis, others)")
uploaded = files.upload()

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("faces_dataset")

print("✅ Dataset extracted to 'faces_dataset/'")

# ============================
# STEP 4: Load and encode known faces
# ============================
known_face_encodings = []
known_face_names = []

dataset_dir = "faces_dataset"

for person_name in os.listdir(dataset_dir):
    person_folder = os.path.join(dataset_dir, person_name)
    if not os.path.isdir(person_folder):
        continue

    for filename in os.listdir(person_folder):
        file_path = os.path.join(person_folder, filename)
        if os.path.isfile(file_path) and file_path.lower().endswith(('.jpg', '.jpeg', '.png')):
            image = face_recognition.load_image_file(file_path)
            encodings = face_recognition.face_encodings(image)
            if encodings:
                for encoding in encodings:  # Add all faces from this image
                    known_face_encodings.append(encoding)
                    known_face_names.append(person_name)
            else:
                print(f"⚠️ No face found in {file_path}, skipping.")

print(f"✅ Encoded {len(known_face_names)} faces from dataset.")

# ============================
# STEP 5: Upload test image
# ============================
print("⬆️ Upload a test image to recognize faces in...")
test_uploaded = files.upload()

test_image_path = list(test_uploaded.keys())[0]

test_image = face_recognition.load_image_file(test_image_path)
face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

# Convert to OpenCV BGR format for display
test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

# ============================
# STEP 6: Run recognition & display
# ============================
for (top, right, b


In [ ]:
# ============================
# STEP 1: Install dependencies
# ============================
!apt-get update
!apt-get install -y cmake
!pip install face_recognition opencv-python

# ============================
# STEP 2: Import libraries
# ============================
import os
import zipfile
import face_recognition
import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# ============================
# STEP 3: Upload & unzip dataset
# ============================
print("⬆️ Upload your faces_dataset.zip (with subfolders: mum, me, sis_pec, sis_mess, bro_wis, others)")
uploaded = files.upload()

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("faces_dataset")

print("✅ Dataset extracted to 'faces_dataset/'")

# ============================
# STEP 4: Load and encode known faces
# ============================
known_face_encodings = []
known_face_names = []

dataset_dir = "faces_dataset"

for person_name in os.listdir(dataset_dir):
    person_folder = os.path.join(dataset_dir, person_name)
    if not os.path.isdir(person_folder):
        continue

    for filename in os.listdir(person_folder):
        file_path = os.path.join(person_folder, filename)
        if os.path.isfile(file_path) and file_path.lower().endswith(('.jpg', '.jpeg', '.png')):
            image = face_recognition.load_image_file(file_path)
            encodings = face_recognition.face_encodings(image)
            if encodings:
                for encoding in encodings:  # Add all faces from this image
                    known_face_encodings.append(encoding)
                    known_face_names.append(person_name)
            else:
                print(f"⚠️ No face found in {file_path}, skipping.")

print(f"✅ Encoded {len(known_face_names)} faces from dataset.")

# ============================
# STEP 5: Upload test image
# ============================
print("⬆️ Upload a test image to recognize faces in...")
test_uploaded = files.upload()

test_image_path = list(test_uploaded.keys())[0]

test_image = face_recognition.load_image_file(test_image_path)
face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

# Convert to OpenCV BGR format for display
test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

# ============================
# STEP 6: Run recognition & display
# ============================
for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)

    if len(known_face_encodings) > 0:
        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        best_match_index = np.argmin(face_distances)
    else:
        face_distances = []
        best_match_index = -1  # no known faces

    if matches and best_match_index != -1 and matches[best_match_index]:
        name = known_face_names[best_match_index]
    else:
        name = "Unknown"

    cv2.rectangle(test_image_cv, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(test_image_cv, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

cv2_imshow(test_image_cv)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("\n🎯 Recognition complete!")


In [ ]:
# ============================
# STEP 1: Install dependencies
# ============================
!apt-get update
!apt-get install -y cmake
!pip install face_recognition opencv-python

# ============================
# STEP 2: Import libraries
# ============================
import os
import zipfile
import face_recognition
import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# ============================
# STEP 3: Upload & unzip dataset
# ============================
print("⬆️ Upload your faces_dataset.zip (with subfolders: mum, me, sis_pec, sis_mess, bro_wis, others)")
uploaded = files.upload()

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("faces_dataset")

print("✅ Dataset extracted to 'faces_dataset/'")

# ============================
# STEP 4: Load and encode known faces
# ============================
known_face_encodings = []
known_face_names = []

dataset_dir = "faces_dataset"

for person_name in os.listdir(dataset_dir):
    person_folder = os.path.join(dataset_dir, person_name)
    if not os.path.isdir(person_folder):
        continue

    for filename in os.listdir(person_folder):
        file_path = os.path.join(person_folder, filename)
        if os.path.isfile(file_path) and file_path.lower().endswith(('.jpg', '.jpeg', '.png')):
            image = face_recognition.load_image_file(file_path)
            encodings = face_recognition.face_encodings(image)
            if encodings:
                for encoding in encodings:  # Add all faces from this image
                    known_face_encodings.append(encoding)
                    known_face_names.append(person_name)
            else:
                print(f"⚠️ No face found in {file_path}, skipping.")

print(f"✅ Encoded {len(known_face_names)} faces from dataset.")

# ============================
# STEP 5: Upload test image
# ============================
print("⬆️ Upload a test image to recognize faces in...")
test_uploaded = files.upload()

test_image_path = list(test_uploaded.keys())[0]

test_image = face_recognition.load_image_file(test_image_path)
face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

# Convert to OpenCV BGR format for display
test_image_cv = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

# ============================
# STEP 6: Run recognition & display
# ============================
for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)

    if len(known_face_encodings) > 0:
        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        best_match_index = np.argmin(face_distances)
    else:
        face_distances = []
        best_match_index = -1  # no known faces

    if matches and best_match_index != -1 and matches[best_match_index]:
        name = known_face_names[best_match_index]
    else:
        name = "Unknown"

    cv2.rectangle(test_image_cv, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(test_image_cv, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

cv2_imshow(test_image_cv)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("\n🎯 Recognition complete!")

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [80.2 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,942 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/u

RuntimeError: Error while calling cudaGetDevice(&the_device_id) in file /tmp/.tmpNZoFGg/sdists-v9/pypi/dlib/19.24.6/mZZaFg42M-YV0ZsyCth_2/src/dlib/cuda/gpu_data.cpp:204. code: 35, reason: CUDA driver version is insufficient for CUDA runtime version